# Tools with MCP ⏰

The Model Context Protocol (MCP) provides a standardized way to connect AI agents to external tools and data sources. Let's connect to an MCP server using `langchain-mcp-adapters`.

## Setup

Load and/or check for needed environmental variables

In [1]:
%%capture
%pip install --upgrade langgraph langchain langchain-ollama langchain-community
%pip install --upgrade langchain-mcp-adapters mcp anyio python-dotenv nest_asyncio

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Ollama server URL - override with OLLAMA_BASE_URL env var
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

In [3]:
# Windows + Jupyter workaround: Jupyter's stderr has no OS file handle on Windows,
# so MCP stdio servers can't inherit it. Send the server's stderr to a log file instead.
import logging
from functools import partial

import langchain_mcp_adapters.sessions as mcp_sessions
from mcp.client.stdio import stdio_client

mcp_server_log = open("mcp_server.log", "w", encoding="utf-8")
mcp_sessions.stdio_client = partial(stdio_client, errlog=mcp_server_log)

# time-mcp prints a startup banner on stdout; hide the harmless parse warning it causes
logging.getLogger("mcp.client.stdio").setLevel(logging.CRITICAL)

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({
    "time": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "time-mcp"],
    }
})

mcp_tools = await mcp_client.get_tools()
print([tool.name for tool in mcp_tools])

['current_time', 'relative_time', 'days_in_month', 'get_timestamp', 'convert_time', 'get_week_year']


Create an agent with the MCP-provided time tools.

In [5]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

chat = ChatOllama(
    model="llama3.2:latest",
    temperature=0.7,
    base_url=OLLAMA_BASE_URL,
)

agent_with_mcp = create_agent(
    model=chat,
    tools=mcp_tools,
    system_prompt="You are a helpful assistant",
)

Ask about the current time in San Francisco.

In [6]:
result = await agent_with_mcp.ainvoke(
    {"messages": [{"role": "user", "content": "What's the time in SF right now?"}]}
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What's the time in SF right now?
================================== Ai Message ==================================
Tool Calls:
  current_time (d9010c16-625b-4aa3-919c-a8adccd5eb25)
 Call ID: d9010c16-625b-4aa3-919c-a8adccd5eb25
  Args:
    format: YYYY-MM-DD HH:mm:ss
    timezone: America/Los_Angeles
================================= Tool Message =================================
Name: current_time

[{'type': 'text', 'text': 'Current UTC time is 2026-10-15 07:04:44, and the time in America/Los_Angeles is 2026-10-15 00:04:44.', 'id': 'lc_a1e8d09d-e474-4e33-9814-68ca613e3c82'}]
================================== Ai Message ==================================

The current time in SF (San Francisco) is 12:04 AM PST.
